# 03 — Prompt Engineering in Code

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/07_AI_Engineering/03_prompt_engineering_in_code.ipynb)

## 📌 What is it?
Prompt engineering is the practice of crafting inputs to LLMs to reliably get desired outputs. In code, it means building dynamic, structured prompts.

## 🧠 Why AI Engineers need this
The quality of your prompts directly determines the quality of your AI product. Systematic prompt engineering is what separates great AI apps from mediocre ones.

In [ ]:
# ── PROMPT TEMPLATES ──
from string import Template
from dataclasses import dataclass
from typing import Optional

@dataclass
class PromptTemplate:
    """Reusable, parameterized prompt template."""
    system: str
    user_template: str
    
    def format(self, **kwargs) -> dict:
        """Fill in template variables and return messages dict."""
        user_content = self.user_template.format(**kwargs)
        return {
            "system": self.system,
            "messages": [{"role": "user", "content": user_content}]
        }

# ── CLASSIFICATION PROMPT ──
classifier_prompt = PromptTemplate(
    system="""You are a precise text classifier. 
Classify the input into exactly one category and respond ONLY with valid JSON.
Categories: positive, negative, neutral""",
    user_template="""Text: {text}

Respond with this exact JSON format:
{{
  "category": "<positive|negative|neutral>",
  "confidence": <0.0-1.0>,
  "reasoning": "<one sentence>"
}}""")

prompt = classifier_prompt.format(text="This product is absolutely amazing!")
print("System:", prompt["system"][:80] + "...")
print("User:",   prompt["messages"][0]["content"])

In [ ]:
# ── FEW-SHOT PROMPTING ──
def build_few_shot_prompt(
    task_description: str,
    examples: list[dict],
    new_input: str
) -> str:
    """Build a few-shot prompt with examples."""
    prompt = f"{task_description}\n\n"
    prompt += "Here are some examples:\n\n"
    
    for i, ex in enumerate(examples, 1):
        prompt += f"Example {i}:\n"
        prompt += f"Input: {ex['input']}\n"
        prompt += f"Output: {ex['output']}\n\n"
    
    prompt += f"Now classify this:\n"
    prompt += f"Input: {new_input}\n"
    prompt += f"Output:"
    return prompt

examples = [
    {"input": "I love this so much!",     "output": "positive"},
    {"input": "This is terrible.",        "output": "negative"},
    {"input": "It arrived on Tuesday.",   "output": "neutral"},
    {"input": "Worst purchase ever.",     "output": "negative"},
    {"input": "Pretty good, I guess.",    "output": "neutral"},
]

prompt = build_few_shot_prompt(
    "Classify customer reviews as positive, negative, or neutral.",
    examples,
    "Exceeded all my expectations!"
)
print(prompt)

In [ ]:
# ── CHAIN-OF-THOUGHT (CoT) ──
def build_cot_prompt(question: str) -> str:
    """Build a chain-of-thought prompt for complex reasoning."""
    return f"""Think through this step by step before giving your final answer.

Question: {question}

Let me think through this:
Step 1: [First consideration]
Step 2: [Second consideration]
Step 3: [Draw conclusion]

Final Answer: [Your answer]"""

print(build_cot_prompt("If I have 3 API calls each taking 0.5s sequentially vs parallel, how much time do I save?"))

In [ ]:
# ── OUTPUT PARSING — extract structured data from LLM responses ──
import json, re
from typing import Optional

def parse_json_response(raw: str) -> Optional[dict]:
    """
    Robustly extract JSON from LLM output.
    LLMs often wrap JSON in markdown code blocks.
    """
    # Try 1: direct parse
    try:
        return json.loads(raw.strip())
    except json.JSONDecodeError:
        pass
    
    # Try 2: extract from ```json ... ``` block
    match = re.search(r'```(?:json)?\s*([\s\S]+?)\s*```', raw)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass
    
    # Try 3: find first { ... } block
    match = re.search(r'\{[\s\S]+\}', raw)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    
    return None

# Test with messy LLM outputs
test_cases = [
    '{"category": "positive", "confidence": 0.92}',
    '```json\n{"category": "negative", "confidence": 0.85}\n```',
    'Sure! Here is the result: {"category": "neutral", "confidence": 0.70}. Hope that helps!'
]

for raw in test_cases:
    result = parse_json_response(raw)
    print(f"✅ Parsed: {result}")

## 🔗 What's Next?
[04 — Embeddings →](04_embeddings.ipynb)